In [19]:

import pandas as pd
import numpy as np

df = pd.read_excel(r"../../Data/raw_data/FintechTrainee.xlsx")


In [20]:
# ISSUE 1 — Duplicate rows (8 duplicate Customer_IDs)

df = df.drop_duplicates(subset='Customer_ID', keep='first')
print(f"After removing duplicate Customer_IDs: {df.shape[0]} rows")


After removing duplicate Customer_IDs: 500 rows


In [21]:
# Note: 3 pairs of different customers share the same NIC number.
shared_nics = ['002727237V', '850611344V', '918354937V']
df['NIC_Duplicate_Flag'] = df['NIC_Number'].isin(shared_nics)
print(f"Shared NIC flag applied to {df['NIC_Duplicate_Flag'].sum()} rows")


Shared NIC flag applied to 6 rows


In [22]:
# ISSUE 2 — Gender formatting inconsistencies
df['Gender'] = df['Gender'].str.strip().str.lower()
df['Gender'] = df['Gender'].map({
    'male': 'Male', 'm': 'Male',
    'female': 'Female', 'f': 'Female'
}).fillna(df['Gender'].str.title())

In [23]:
# ISSUE 3 — Account_Status casing inconsistencies
df['Account_Status'] = df['Account_Status'].str.strip().str.title()

In [24]:
district_fixes = {
    'Columbo': 'Colombo',
    'Kanady': 'Kandy',
    'Gale': 'Galle',
    'Jafna': 'Jaffna'
}
df['District'] = df['District'].replace(district_fixes)

In [25]:
# ISSUE 5 — Province re-derivation from corrected District
district_province_map = {
    'Colombo': 'Western', 'Gampaha': 'Western', 'Kalutara': 'Western',
    'Kandy': 'Central', 'Matale': 'Central', 'Nuwara Eliya': 'Central',
    'Galle': 'Southern', 'Matara': 'Southern', 'Hambantota': 'Southern',
    'Jaffna': 'Northern', 'Kilinochchi': 'Northern', 'Mannar': 'Northern',
    'Mullaitivu': 'Northern', 'Vavuniya': 'Northern',
    'Trincomalee': 'Eastern', 'Batticaloa': 'Eastern', 'Ampara': 'Eastern',
    'Kurunegala': 'North Western', 'Puttalam': 'North Western',
    'Anuradhapura': 'North Central', 'Polonnaruwa': 'North Central',
    'Badulla': 'Uva', 'Monaragala': 'Uva',
    'Ratnapura': 'Sabaragamuwa', 'Kegalle': 'Sabaragamuwa'
}
df['Province'] = df['District'].map(district_province_map).fillna(df['Province'])


In [26]:
# ISSUE 6 — Sentinel / placeholder values
df.loc[df['Savings_Balance'] == 999_999_999, 'Savings_Balance'] = np.nan
df.loc[df['Monthly_Withdrawal_Avg'] >= 8_000_000, 'Net_Monthly_Flow'] = np.nan
df.loc[df['Monthly_Withdrawal_Avg'] >= 8_000_000, 'Monthly_Withdrawal_Avg'] = np.nan


In [27]:
# ISSUE 7 — Negative savings balances (4 customers)
df['Savings_Balance_Invalid'] = df['Savings_Balance'] < 0
df.loc[df['Savings_Balance'] < 0, 'Savings_Balance'] = np.nan
print(f"Savings balances set to NaN (negative/sentinel): {df['Savings_Balance'].isna().sum()} rows")


Savings balances set to NaN (negative/sentinel): 11 rows


In [28]:
# ISSUE 8 — Impossible ages (3 customers)
impossible_ages = [4, 134, 150]
df['Age_Invalid'] = df['Age'].isin(impossible_ages)
df.loc[df['Age_Invalid'], 'Age'] = np.nan
print(f"Ages set to NaN (impossible): {df['Age_Invalid'].sum()} rows")


Ages set to NaN (impossible): 3 rows


In [29]:
# ISSUE 9 — Loan field contradictions (5 customers)
loan_contradiction = (
    ((df['Has_Loan'] == 'Yes') & df['Loan_Amount'].isna()) |
    ((df['Has_Loan'] == 'No') & df['Outstanding_Loan_Balance'].notna())
)
df['Loan_Data_Inconsistent'] = loan_contradiction
print(f"Loan data inconsistency flags: {df['Loan_Data_Inconsistent'].sum()} rows")


Loan data inconsistency flags: 5 rows


In [30]:
# ISSUE 10 — Number_of_Products miscounts (6 customers)
for col in ['Has_Loan', 'Has_Fixed_Deposit', 'Has_Insurance', 'Has_Mobile_Wallet']:
    df[col] = df[col].fillna('No')

df['Number_of_Products'] = (
    1 +
    (df['Has_Loan'] == 'Yes').astype(int) +
    (df['Has_Fixed_Deposit'] == 'Yes').astype(int) +
    (df['Has_Insurance'] == 'Yes').astype(int) +
    (df['Has_Mobile_Wallet'] == 'Yes').astype(int)
)


In [31]:
# ISSUE 11 — Missing Referral_Source (252 rows)
df['Referral_Source'] = df['Referral_Source'].fillna('None')

In [32]:
df_clean = df.copy()

In [ ]:
# TO save the cleaned file 
df_clean.to_excel('FinSight_Cleaned.xlsx', index=False)

In [35]:
df_clean.to_excel(r"../../Data/cleaned_data/FintechTrainee_Cleaned.xlsx", index=False)